In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("HomeCredit_Serving_Layer") \
    .enableHiveSupport() \
    .getOrCreate()

In [37]:
raw_app = spark.read.parquet("/user/student/home_credit/raw/application_train")
raw_prev = spark.read.parquet("/user/student/home_credit/raw/previous_application")
raw_inst = spark.read.parquet("/user/student/home_credit/raw/installments_payments")
raw_bureau = spark.read.parquet("/user/student/home_credit/raw/bureau")

print("Raw Parquet Data loaded successfully from HDFS!")

Raw Parquet Data loaded successfully from HDFS!


In [ ]:
# 1. Application Train
df_app = raw_app.select(
    F.col("SK_ID_CURR").cast("int"),
    F.col("TARGET").cast("int"),
    F.col("NAME_CONTRACT_TYPE").cast("string"),
    F.col("DAYS_BIRTH").cast("int"),
    F.col("OCCUPATION_TYPE").cast("string"),
    F.col("NAME_EDUCATION_TYPE").cast("string"),
    F.col("NAME_FAMILY_STATUS").cast("string"),
    F.col("NAME_HOUSING_TYPE").cast("string"),
    F.col("NAME_INCOME_TYPE").cast("string"),
    F.col("FLAG_OWN_REALTY").cast("string"),
    F.col("AMT_INCOME_TOTAL").cast("double"),
    F.col("AMT_CREDIT").cast("double"),
    F.col("AMT_ANNUITY").cast("double"),
    F.col("AMT_GOODS_PRICE").cast("double"),
    F.col("DAYS_EMPLOYED").cast("int"),
    F.col("CODE_GENDER").cast("string"),
    F.col("FLAG_OWN_CAR").cast("string")
)

# 2. Previous Applications
df_prev = raw_prev.select(
    F.col("SK_ID_PREV").cast("int"),
    F.col("SK_ID_CURR").cast("int"),
    F.col("NAME_CONTRACT_STATUS").cast("string"),
    F.col("AMT_CREDIT").cast("double")
)

# 3. Installments Payments
df_inst = raw_inst.select(
    F.col("SK_ID_PREV").cast("int"),
    F.col("SK_ID_CURR").cast("int"),
    F.col("DAYS_INSTALMENT").cast("double"),
    F.col("DAYS_ENTRY_PAYMENT").cast("double"),
    F.col("AMT_INSTALMENT").cast("double"),
    F.col("AMT_PAYMENT").cast("double"),
    F.col("NUM_INSTALMENT_VERSION").cast("double"),
    F.col("NUM_INSTALMENT_NUMBER").cast("int"),
)

# 4. Bureau Data
df_bureau = raw_bureau.select(
    F.col("SK_ID_CURR").cast("int"),
    F.col("CREDIT_ACTIVE").cast("string"),
    F.col("AMT_CREDIT_SUM_DEBT").cast("double"),
    F.col("AMT_CREDIT_SUM_OVERDUE").cast("double"),
    F.col("AMT_CREDIT_MAX_OVERDUE").cast("double"),
    F.col("AMT_CREDIT_SUM").cast("double"),
    F.col("SK_ID_BUREAU").cast("int"),
    F.col("CREDIT_DAY_OVERDUE").cast("int"),
)

print("Data selected successfully!")

Data selected successfully!


In [39]:
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

def clean_empty_strings(df):
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    
    for c in string_cols:
        df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c)))
        
    return df

df_app = clean_empty_strings(df_app)
df_prev = clean_empty_strings(df_prev)
df_inst = clean_empty_strings(df_inst)
df_bureau = clean_empty_strings(df_bureau)

print("Transformations applied")

Transformations applied


In [ ]:
clean_app = df_app.dropDuplicates(["SK_ID_CURR"]).filter(F.col("SK_ID_CURR").isNotNull())
clean_prev = df_prev.dropDuplicates(["SK_ID_PREV"]).filter(F.col("SK_ID_CURR").isNotNull())
clean_inst = df_inst.dropDuplicates().filter(F.col("SK_ID_CURR").isNotNull())
clean_bureau = df_bureau.dropDuplicates().filter(F.col("SK_ID_CURR").isNotNull())

print("Data cleaning completed successfully!")

Data cleaning completed successfully!


In [46]:
#clean_app.printSchema()
#clean_prev.printSchema()
#clean_inst.printSchema()
#clean_bureau.printSchema()

#clean_app.show(20)
#clean_prev.show(20)
#clean_inst.show(20)
#clean_bureau.show(20)

In [ ]:
clean_app_fixed = clean_app \
    .withColumn("DAYS_EMPLOYED", F.when(F.col("DAYS_EMPLOYED") == 365243, None).otherwise(F.col("DAYS_EMPLOYED"))) \
    .withColumn("OCCUPATION_TYPE", F.when(F.col("NAME_INCOME_TYPE") == "Pensioner", "Retired").otherwise(F.col("OCCUPATION_TYPE"))) \
    .withColumn("DAYS_BIRTH", F.abs(F.col("DAYS_BIRTH"))) \
    .withColumn("DAYS_EMPLOYED", F.abs(F.col("DAYS_EMPLOYED")))

In [ ]:
clean_inst_fixed = clean_inst \
    .withColumn("DAYS_INSTALMENT", F.abs(F.col("DAYS_INSTALMENT"))) \
    .withColumn("DAYS_ENTRY_PAYMENT", F.abs(F.col("DAYS_ENTRY_PAYMENT")))

In [ ]:
clean_bureau_fixed = clean_bureau \
    .withColumn("AMT_CREDIT_SUM", F.when(F.col("AMT_CREDIT_SUM") <= 0.0, None).otherwise(F.col("AMT_CREDIT_SUM"))) \
    .fillna(0.0, subset=["CREDIT_DAY_OVERDUE","AMT_CREDIT_SUM","AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "AMT_CREDIT_MAX_OVERDUE"])

In [ ]:
clean_app_fixed.write.mode("overwrite").parquet("/user/student/home_credit/staging/application_train") 
clean_prev.write.mode("overwrite").parquet("/user/student/home_credit/staging/previous_application")
clean_inst_fixed.write.mode("overwrite").parquet("/user/student/home_credit/staging/installments_payments") 
clean_bureau_fixed.write.mode("overwrite").parquet("/user/student/home_credit/staging/bureau")
print("all done")

all done


In [ ]:
fact_app_features = (
    clean_app_fixed
    .withColumn("DTI", F.when(F.col("AMT_INCOME_TOTAL") > 0, F.round(F.col("AMT_CREDIT") / F.col("AMT_INCOME_TOTAL"), 4)).otherwise(None))
    .withColumn("Annuity_to_Income", F.when(F.col("AMT_INCOME_TOTAL") > 0, F.round(F.col("AMT_ANNUITY") / F.col("AMT_INCOME_TOTAL"), 4)).otherwise(None))
    .withColumn("LTV", F.when(F.col("AMT_GOODS_PRICE") > 0, F.round(F.col("AMT_CREDIT") / F.col("AMT_GOODS_PRICE"), 4)).otherwise(None))
    .withColumn("Employed_to_Age", F.when(F.col("DAYS_EMPLOYED").isNotNull(), F.round(F.col("DAYS_EMPLOYED") / F.col("DAYS_BIRTH"), 4)).otherwise(0.0))
)
#fact_app_features.select("SK_ID_CURR", "AMT_CREDIT", "AMT_INCOME_TOTAL", "DTI" , "LTV" , "Annuity_to_Income" , "Employed_to_Age").show(20)

In [ ]:
prev_features = clean_prev.groupBy("SK_ID_CURR").agg(
    F.count("SK_ID_PREV").alias("Prev_App_Count"),
    F.round(
        F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Approved", 1).otherwise(0)) / F.count("SK_ID_PREV"),
        4
    ).alias("Approved_App_Ratio"),
    F.round(
        F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Refused", 1).otherwise(0)) / F.count("SK_ID_PREV"),
        4
    ).alias("Refused_App_Ratio"),
    F.round(F.avg("AMT_CREDIT"), 2 ).alias("Avg_Prev_Credit")
        
)
#prev_features.show(5)

In [ ]:
inst_level = (
    clean_inst_fixed
    .groupBy(
        "SK_ID_CURR",
        "SK_ID_PREV",
        "NUM_INSTALMENT_VERSION",
        "NUM_INSTALMENT_NUMBER"
    )
    .agg(
        F.min("DAYS_INSTALMENT").alias("DAYS_INSTALMENT"),
        F.max("DAYS_ENTRY_PAYMENT").alias("DAYS_ENTRY_PAYMENT"),
        F.max("AMT_INSTALMENT").alias("AMT_INSTALMENT"),
        F.sum("AMT_PAYMENT").alias("AMT_PAYMENT")
    )
)

inst_level = (
    inst_level
    .withColumn(
        "Days_Past_Due",
        F.greatest(F.col("DAYS_INSTALMENT") - F.col("DAYS_ENTRY_PAYMENT"), F.lit(0.0))
    )
    .withColumn(
        "Is_Late",
        F.when(F.col("Days_Past_Due") > 0, 1).otherwise(0)
    )
    .withColumn(
        "Underpaid_Amount",
        F.greatest(F.col("AMT_INSTALMENT") - F.col("AMT_PAYMENT"), F.lit(0.0))
    )
)

inst_agg = (
    inst_level
    .groupBy("SK_ID_CURR")
    .agg(
        # Feature 1: Total_Days_Past_Due
        F.round(F.sum("Days_Past_Due"), 2).alias("Total_Days_Past_Due"),
        
        # Feature 2: Num_Late_Payments
        F.sum("Is_Late").alias("Num_Late_Payments"),
        
        # Feature 3: Avg_Days_Past_Due
        F.round(F.avg("Days_Past_Due"), 2).alias("Avg_Days_Past_Due"),
        
        # Feature 4: Max_Days_Past_Due
        F.round(F.max("Days_Past_Due"), 2).alias("Max_Days_Past_Due"),
        
        # Feature 5: Total_Underpaid
        F.round(F.sum("Underpaid_Amount"), 2).alias("Total_Underpaid"),
        
        # Intermediate sums needed for Payment_Ratio
        F.sum("AMT_PAYMENT").alias("SUM_AMT_PAYMENT"),
        F.sum("AMT_INSTALMENT").alias("SUM_AMT_INSTALMENT")
    )
)

inst_features = (
    inst_agg
    .withColumn(
        "Payment_Ratio",
        F.when(
            F.col("SUM_AMT_INSTALMENT") > 0,
            F.round(F.col("SUM_AMT_PAYMENT") / F.col("SUM_AMT_INSTALMENT"), 4)
        ).otherwise(None)
    )
    .drop("SUM_AMT_PAYMENT", "SUM_AMT_INSTALMENT") # Drop intermediate columns
)

#inst_features.show(5)

In [ ]:
bureau_features = (
    clean_bureau_fixed
    .groupBy("SK_ID_CURR")
    .agg(
        # Total number of bureau loans
        F.countDistinct("SK_ID_BUREAU").alias("Bureau_Credit_Count"),
        
        # Number of active loans
        F.sum(
            F.when(F.col("CREDIT_ACTIVE") == "Active", 1).otherwise(0)
        ).alias("Bureau_Active_Loans"),
        
        # Total remaining external debt
        F.round(F.sum("AMT_CREDIT_SUM_DEBT"), 2).alias("Total_External_Debt"),
        
        # Total amount currently overdue
        F.round(F.sum("AMT_CREDIT_SUM_OVERDUE"), 2).alias("Total_External_Overdue"),
        
        # Max overdue amount recorded
        F.round(F.max("AMT_CREDIT_MAX_OVERDUE"), 2).alias("Max_External_Overdue"),
        
        # Max overdue days recorded
        F.max("CREDIT_DAY_OVERDUE").alias("Bureau_Max_Days_Overdue"),
        
        # Helper column for credit sum
        F.sum("AMT_CREDIT_SUM").alias("SUM_AMT_CREDIT_SUM")
    )
    # Ratio of active loans to total loans
    .withColumn(
        "Active_Credit_Ratio",
        F.when(
            F.col("Bureau_Credit_Count") > 0,
            F.round(F.col("Bureau_Active_Loans") / F.col("Bureau_Credit_Count"), 4)
        ).otherwise(0.0)
    )
    # Ratio of total debt to total credit
    .withColumn(
        "Debt_to_Credit_Bureau",
        F.when(
            F.col("SUM_AMT_CREDIT_SUM") > 0,
            F.round(F.col("Total_External_Debt") / F.col("SUM_AMT_CREDIT_SUM"), 4)
        ).otherwise(0.0)
    )
    .drop("SUM_AMT_CREDIT_SUM")
)

#bureau_features.show(5)

In [ ]:
avg_payment_df = inst_level.groupBy("SK_ID_CURR").agg(
    F.round(F.avg("AMT_PAYMENT"), 2).alias("Avg_Historical_Payment")
)

fact_loan = (
    fact_app_features
    .join(prev_features, on="SK_ID_CURR", how="left")
    .join(inst_features, on="SK_ID_CURR", how="left")
    .join(bureau_features, on="SK_ID_CURR", how="left")
    .join(avg_payment_df, on="SK_ID_CURR", how="left")
)

fact_loan = (
    fact_loan
    
    # Feature 1: Total_Overall_Debt
    .withColumn(
        "Total_Overall_Debt",
        F.col("AMT_CREDIT") + F.coalesce(F.col("Total_External_Debt"), F.lit(0.0))
    )
    
    # Feature 2: External_DTI
    .withColumn(
        "External_DTI",
        F.when(
            F.col("AMT_INCOME_TOTAL") > 0,
            F.round(F.coalesce(F.col("Total_External_Debt"), F.lit(0.0)) / F.col("AMT_INCOME_TOTAL"), 4)
        ).otherwise(0.0)
    )
    
    # Feature 3: Current_vs_Prev_Credit
    .withColumn(
        "Current_vs_Prev_Credit",
        F.when(
            F.col("Avg_Prev_Credit").isNotNull() & (F.col("Avg_Prev_Credit") > 0),
            F.round(F.col("AMT_CREDIT") / F.col("Avg_Prev_Credit"), 4)
        ).otherwise(None)
    )
    
    # Feature 4: Annuity_vs_Historical_Payment
    .withColumn(
        "Annuity_vs_Historical_Payment",
        F.when(
            F.col("Avg_Historical_Payment").isNotNull() & (F.col("Avg_Historical_Payment") > 0),
            F.round(F.col("AMT_ANNUITY") / F.col("Avg_Historical_Payment"), 4)
        ).otherwise(None)
    )
)

#print(f"Total rows in Fact_Loan: {fact_loan.count()}")
#print(f"Distinct SK_ID_CURR in Fact_Loan: {fact_loan.select('SK_ID_CURR').distinct().count()}")


In [ ]:
dim_customer = clean_app_fixed.select(
    "SK_ID_CURR",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "NAME_CONTRACT_TYPE",
    "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "NAME_INCOME_TYPE",
    "FLAG_OWN_REALTY"
)

fact_loan_final = fact_loan.drop(
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "NAME_CONTRACT_TYPE",
    "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "NAME_INCOME_TYPE",
    "FLAG_OWN_REALTY"
)

spark.sql("CREATE DATABASE IF NOT EXISTS home_credit_dw")

(dim_customer.write 
    .mode("overwrite") 
    .format("parquet") 
    .option("compression", "snappy") 
    .option("path", "/user/student/home_credit/warehouse/dim_customer") 
    .saveAsTable("home_credit_dw.dim_customer"))

(fact_loan_final.write 
    .mode("overwrite") 
    .format("parquet") 
    .option("compression", "snappy") 
    .option("path", "/user/student/home_credit/warehouse/fact_loan") 
    .saveAsTable("home_credit_dw.fact_loan"))

print("Data Warehouse tables created.")
